In [ ]:
library(plyr)
library(dplyr)
library(tidyr)
library(lme4)
library(lmerTest)
library(interactions)
library(ggplot2)
library(emmeans)
library(rempsyc)

# TF measures predicted by acc * soc * age

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/FB/", analysis_path, session))
if (!dir.exists(pic_path)) {
  dir.create(pic_path, recursive = TRUE)
}
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
feedback_counts <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*feedback*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/FB/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'power_early',
    'ITPS_early',
    'ICPS_early_DLPFC_collapsed',
    'ICPS_early_MOTOR_collapsed',
    'ICPS_early_OCC_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

feedback_counts <- feedback_counts %>%
  pivot_wider(
    names_from = Category,
    values_from = Count,
    names_prefix = "cat"
  )

tf_data <- tf_data %>%
  left_join(
    feedback_counts %>%
      mutate(
        sub = as.factor(sub),
        soc = factor(soc, levels = c(0, 1), labels = c("NS", "S"))
      ) %>%
      select(sub, soc, cat1, cat2, cat3),
    by = c("sub", "soc")
  )

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

tf_data$cat1 <- tf_data$cat1 / 10
tf_data$cat2 <- tf_data$cat2 / 10
tf_data$cat3 <- tf_data$cat3 / 10

tf_data$cat1 <- scale(tf_data$cat1)
tf_data$cat2 <- scale(tf_data$cat2)
tf_data$cat3 <- scale(tf_data$cat3)

In [ ]:
head(feedback_counts)

In [ ]:
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
filter(tf_data, sub == 3000314)$age_m[1] # now 154

In [ ]:
filter(tf_data, sub == 3000252)$age_m[1] # now 168

In [ ]:
filter(tf_data, sub == 3000269)$age_m[1] # now 161

In [ ]:
filter(tf_data, sub == 3000014)$age_m[1] # now 160

In [ ]:
filter(tf_data, sub == 3000014)

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
  "power_early", 
  "ITPS_early", 
  "ICPS_early_DLPFC_collapsed", 
  "ICPS_early_MOTOR_collapsed", 
  "ICPS_early_OCC_collapsed"
)

# 2. Define the static right-hand side of your model formula
predictors <- "acc * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}


In [ ]:
outcome <- "ICPS_early_DLPFC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

### Plot acc*age ICPS

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

for (outcome in c("power_early", "ITPS_early")) {

    predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
    f <- as.formula(paste(outcome, "~", predictors))
    cat(sprintf("Fitting model for %s...\n", outcome))
    
    model <- lmer(f, data = tf_data)
    # print(summary(model))
    x_label = "Age"    
    if (outcome == "power_early") {
        y_label = "Power"
        label = "power"
    } else if (outcome == "ITPS_early") {
        y_label = "ITPS"
        label = "ITPS"
    }
    
    points = FALSE
    if (points == FALSE) {
        png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=10, height=4, units="in", res=600)
    } else {
        png(file=sprintf("%s/%s_stats.png", pic_path, label), width=10, height=4, units="in", res=600)
    }
    interact_plot(model,
                  pred = "age_m",
                  modx = "acc",
                  interval = 1,
                  y.label = y_label,
                  x.label = x_label,
                  legend.main = "Accuracy",
                  colors = c("red", "blue"),
                  plot.points = points) +
    theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
      scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
    theme(
      text = element_text(size = 14),                              # Change the base font size
      axis.title = element_text(size = 16),                        # Change the axis title font size
      axis.text = element_text(size = 14),                         # Change the axis text font size
      legend.text = element_text(size = 14),                       # Change the legend text font size
      legend.title = element_text(size = 14), 
      plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
    )
    dev.off()
}

In [ ]:
for (outcome in c("ICPS_early_DLPFC_collapsed", "ICPS_early_MOTOR_collapsed")) {

    predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
    f <- as.formula(paste(outcome, "~", predictors))
    cat(sprintf("Fitting model for %s...\n", outcome))
    
    model <- lmer(f, data = tf_data)
    
    x_label = "Age"    
    if (outcome == "ICPS_early_DLPFC_collapsed") {
        y_label = "Frontolateral ICPS"
        label = "ICPS_DLPFC"
    } else if (outcome == "ICPS_early_MOTOR_collapsed") {
        y_label = "Midlateral ICPS"
        label = "ICPS_MOTOR"
    }
    
    points = TRUE
    file_name <- if (!points) sprintf("%s/%s_stats_no_points.png", pic_path, label) else sprintf("%s/%s_stats.png", pic_path, label)

    # Assign plot to an object
    p <- interact_plot(model,
                  pred = "age_m",
                  modx = "acc",
                  interval = 1,
                  y.label = y_label,
                  x.label = x_label,
                  legend.main = "Accuracy",
                  colors = c("red", "blue"),
                  plot.points = points) +
        theme_minimal() +  
        scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
        scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
        theme(
          text = element_text(size = 14),                                
          axis.title = element_text(size = 16),                          
          axis.text = element_text(size = 14),                           
          legend.text = element_text(size = 14),                         
          legend.title = element_text(size = 14), 
            legend.position = "none",
          plot.title = element_text(size = 18, face = "bold")            
        )
    
    # ggsave handles printing and rendering automatically
    ggsave(filename = file_name, plot = p, width = 5, height = 4, units = "in", dpi = 600)
}

In [ ]:
pic_path

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome = "power_early"
outcome = "ITPS_early"

predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
    
model <- lmer(f, data = tf_data)
# print(summary(model))
x_label = "Age"    
if (outcome == "power_early") {
    y_label = "Power"
    label = "power"
} else if (outcome == "ITPS_early") {
    y_label = "ITPS"
    label = "ITPS"
}

points = FALSE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=10, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=10, height=4, units="in", res=600)
}
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = points) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14), 
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome <- "ICPS_early_DLPFC_collapsed"
# outcome <- "ICPS_early_MOTOR_collapsed"
outcome <- "ITPS_early"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

# y_label = "Frontolateral ICPS"
# y_label = "Midlateral ICPS"
# y_label = "Power"
y_label = "ITPS"
x_label = "Age"
# label = "ICPS_DLPFC"
# label = "ICPS_MOTOR"
# label = "power"
label = "ITPS"

points = FALSE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=5, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
}
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = points) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14), 
        legend.position = "none", # Change the legend title font size# Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

### Plot acc*age*soc ICPS

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome <- "ICPS_early_DLPFC_collapsed"
outcome <- "ICPS_early_OCC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

# y_label = "Frontolateral ICPS"
y_label = "Posterolateral ICPS"
x_label = "Age"
# label = "ICPS_DLPFC"
label = "ICPS_OCC"

points = TRUE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=9, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=4, units="in", res=600)
}
cond_labels = c("Non-social", "Social")

interact_plot(model,
              pred = "age_m",
              modx = "acc",
              mod2="soc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              colors = c("red", "blue"),
              legend.main = "Accuracy",
              mod2.labels = cond_labels,
              plot.points = points) +
  theme_minimal() +
  theme(
    text = element_text(size = 20),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    legend.text = element_text(size = 14),                       # Change the legend text font size
    legend.title = element_text(size = 14),                     # Change the legend title font size
  legenbbd.position = "none",
    plot.title = element_text(size = 20, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-1, 1, by = 1)) + 
  scale_linetype_manual(values = c("Error" = "solid", "Correct" = "dashed"))
dev.off()

## Posthocs

In [ ]:
# outcome <- "ICPS_early_DLPFC_collapsed"
# outcome <- "ICPS_early_MOTOR_collapsed"
# outcome <- "power_early"
outcome <- "ITPS_early"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
model <- lmer(f, data = tf_data)
simple_slopes <- emtrends(model, ~ acc, var = "age_m")
summary(model)

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE), adjust = 'fdr')

# Calculate the difference between the slopes across levels of 'acc'
slope_diffs <- pairs(simple_slopes)

# View the differences and significance tests
summary(slope_diffs, infer = c(TRUE, TRUE))

library(broom)
library(knitr)

# tidy() converts the objects into clean tibbles
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)
tidy_diffs <- tidy(slope_diffs, conf.int = TRUE)

cat("\n--- Simple Slopes ---\n")
print(kable(tidy_slopes, digits = 3, format = "simple"))

cat("\n--- Slope Differences (e.g., Error - Correct) ---\n")
print(kable(tidy_diffs, digits = 3, format = "simple"))

In [ ]:
outcome <- "ICPS_early_OCC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
model <- lmer(f, data = tf_data)


# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc | soc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "soc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "acc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

### Plot

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Power"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

### Plot

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "ITPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

In [ ]:
library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

### Plot

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

### Plot

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Midlateral ICPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),
    legend.position = "none", # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

### Plot

In [ ]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Posterolateral ICPS"
x_label = "Age"
cond_labels = c("Non-social", "Social")

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              mod2="soc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              colors = c("red", "blue"),
              legend.main = "Accuracy",
              mod2.labels = cond_labels,
              plot.points = T) +
  theme_minimal() +
  theme(
    text = element_text(size = 20),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    legend.text = element_text(size = 14),                       # Change the legend text font size
    legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 20, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-1, 1, by = 1))
  # scale_linetype_manual(values = c("solid", "solid"))
dev.off()

# DDM

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/FB", analysis_path, session))
if (!dir.exists(pic_path)) {
  dir.create(pic_path, recursive = TRUE)
}
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
feedback_counts <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*feedback*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/FB", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'reversed_ratio_diff',
    'a_diff',
    'p_diff',
    'ter_diff',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

feedback_counts <- feedback_counts %>%
  pivot_wider(
    names_from = Category,
    values_from = Count,
    names_prefix = "cat"
  )

tf_data <- tf_data %>%
  left_join(
    feedback_counts %>%
      mutate(
        sub = as.factor(sub),
        soc = factor(soc, levels = c(0, 1), labels = c("NS", "S"))
      ) %>%
      select(sub, soc, cat1, cat2, cat3),
    by = c("sub", "soc")
  )

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

tf_data$cat1 <- tf_data$cat1 / 10
tf_data$cat2 <- tf_data$cat2 / 10
tf_data$cat3 <- tf_data$cat3 / 10

tf_data$cat1 <- scale(tf_data$cat1)
tf_data$cat2 <- scale(tf_data$cat2)
tf_data$cat3 <- scale(tf_data$cat3)

## Full models (all three ICPS)

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_full")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## DLPFC-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_dlpfc")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## MOTOR-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_MOTOR_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_motor")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## OCC-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_occ")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## Plots

In [ ]:
outcome <- "reversed_ratio_diff"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "ratio_reversed"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = reversed_ratio_diff))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

In [ ]:
outcome <- "a_diff"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + cat2 + cat3 + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "a_diff"
plot_label <- bquote(atop("Response Caution", alpha ~ "(Post-error - Post-correct)"))

png(file=sprintf("%s/%s_stats_all_three.png", pic_path, label), width=9, height=5, units="in", res=600)
interact_plot(model, pred = "ICPS_early_DLPFC_diff_collapsed", modx = "age_m", interval = 1, dodge.width = 0.2,
              y.label = plot_label,
              x.label = "Midfrontal-Frontolateral ICPS (Error - Correct)",
              legend.main = "Age",
              plot.points = T) + 
  theme_minimal() +
  theme(
    text = element_text(size = 14), # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    # legend.text = element_text(size = 14),                       # Change the legend text font size
    # legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 18, face = "bold")         # Change the plot title font size and make it bold
  ) + theme(legend.position = "none") +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) + theme(legend.position = "bottom")
dev.off()

In [ ]:
library(emmeans)

# 1. Directly specify the standardized values (-1 SD, Mean, +1 SD)
age_points <- list(age_m = c(-1, 0, 1))

# 2. Compute the simple slopes
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 3. View the slopes with 95% Confidence Intervals and p-values
summary(slopes, infer = c(TRUE, TRUE))

In [ ]:
library(emmeans)

# 1. Compute the slopes at the standardized age points
age_points <- list(age_m = c(-1, 0, 1))
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 2. Compute paired comparisons between the slopes
pairwise_diffs <- pairs(slopes)

# 3. View the results
summary(pairwise_diffs, infer = c(TRUE, TRUE))

# Updated raw behav measures

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'pea',
    'peri_rt',
    'pes',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
# tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

## Full models (all three ICPS)

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_full")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

In [ ]:
outcome <- "pea"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pea"
plot_label <- "Post-Error Accuracy"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = pea))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

In [ ]:
outcome <- "pes"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pes_fl"
plot_label <- "Post-Error Slowing"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_DLPFC_diff_collapsed, y = pes))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Frontolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

In [ ]:
outcome <- "pes"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pes_ml"
plot_label <- "Post-Error Slowing"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_MOTOR_diff_collapsed, y = pes))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Midlateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

## DLPFC-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_dlpfc")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## MOTOR-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_MOTOR_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_motor")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

## OCC-only

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_occ")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

In [ ]:
source(sprintf("%s/code/statistics/combine_docs.R", analysis_path))
combine_docs(table_output_path,
            order_pattern = c(
                "power",
                "ITPS",
                "ICPS",
                "ratio_diff_ms_full",
                "ratio_",
                "a_diff_ms_full",
                "a_diff_",
                "pea_ms_full",
                "pea_",
                "peri_rt_ms_full",
                "peri_rt_",
                "pes_ms_full",
                "pes_"
            )
            )

# Behavior

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}

tf_data <- subset(tf_data, tf_data$acc == 1)
tf_data <- select(tf_data, -acc)

id_cols <- c('sub', 'age_m', 'sex', 'soc', 'dp_inperson')
data_cols <- c(
    'acc_con',
    'acc_incon',
    'rt_con',
    'rt_incon'
)

tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data <- tf_data %>%
  pivot_longer(
    cols = c(acc_con, acc_incon, rt_con, rt_incon),
    names_to = c(".value", "congruency"),
    names_sep = "_"
  ) %>%
  mutate(
    congruency = factor(congruency, levels = c("con", "incon")),
    sub = as.factor(sub),
    soc = as.factor(soc)
  )

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$congruency <- as.factor(tf_data$congruency)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", "acc", "rt")] <- as.data.frame(scale(tf_data[c("age_m", "acc", "rt")]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acongruency <- revalue(tf_data$congruency, c("incon" = "Incongruent", "con" = "Congruent"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$congruency) <- -rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
  "acc",
    "rt"
)

# 2. Define the static right-hand side of your model formula
predictors <- "congruency * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

In [ ]:
outcome <- "rt"

# 2. Define the static right-hand side of your model formula
predictors <- "congruency * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
model <- lmer(f, data = tf_data)

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE), adjust = 'fdr')

# Calculate the difference between the slopes across levels of 'acc'
slope_diffs <- pairs(simple_slopes)

# View the differences and significance tests
summary(slope_diffs, infer = c(TRUE, TRUE))

library(broom)
library(knitr)

# tidy() converts the objects into clean tibbles
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)
tidy_diffs <- tidy(slope_diffs, conf.int = TRUE)

cat("\n--- Simple Slopes ---\n")
print(kable(tidy_slopes, digits = 3, format = "simple"))

cat("\n--- Slope Differences (e.g., Error - Correct) ---\n")
print(kable(tidy_diffs, digits = 3, format = "simple"))